### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit", # Llama 3.2 vision support
    "unsloth/Llama-3.2-11B-Vision-bnb-4bit",
    "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit", # Can fit in a 80GB card!
    "unsloth/Llama-3.2-90B-Vision-bnb-4bit",

    "unsloth/Pixtral-12B-2409-bnb-4bit",              # Pixtral fits in 16GB!
    "unsloth/Pixtral-12B-Base-2409-bnb-4bit",         # Pixtral base model

    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",          # Qwen2 VL support
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit",

    "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",      # Any Llava variant works!
    "unsloth/llava-1.5-7b-hf-bnb-4bit",
] # More models at https://huggingface.co/unsloth

==((====))==  Unsloth 2025.10.2: Fast Qwen3_Vl patching. Transformers: 4.57.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [ ]:
import os
os.environ["TORCHDYNAMO_DISABLE"]    = "1"   # no torch.compile / Inductor
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"   # and don't let Unsloth compile either
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers   = False,   # match the 30B's raw extraction; language only
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules     = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = 42,
)

## Step 2 — load our dataset + convert to Unsloth's vision format
Our examples are `{image: PIL, messages: [...]}` where the user message already carries the
canonical prompt text and an image placeholder. Reshape to Unsloth conversations (image
inlined). **Subset to 300 for the proof run.**

In [ ]:
from datasets import load_dataset
from huggingface_hub import login; login()   # paste a (read) token

raw = load_dataset("MitchMooney/para-vlm-extraction", split="train")
proof = raw.select(range(300))               # PROOF; use all of `raw` for the full run

def text_of(messages, role):
    return next(c["text"] for m in messages if m["role"] == role
               for c in m["content"] if c["type"] == "text")

def to_conv(ex):
    return {"messages": [
        {"role": "user", "content": [
            {"type": "text",  "text": text_of(ex["messages"], "user")},
            {"type": "image", "image": ex["image"]},
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": text_of(ex["messages"], "assistant")},
        ]},
    ]}

train_conv = [to_conv(ex) for ex in proof]
print(len(train_conv), "examples; sample target:", train_conv[0]["messages"][1]["content"][0]["text"][:120])

Generating train split:   0%|          | 0/68686 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7632 [00:00<?, ? examples/s]

## Step 3 — train (proof: ~30 steps)

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_conv,
    args = SFTConfig(
        # PROOF (300 ex): grad_accum=4, max_steps=30, warmup_steps=5, logging_steps=1.
        # FULL run (all 2611 ex) below — drop `proof = raw.select(range(300))` to `proof = raw`:
        per_device_train_batch_size = 1, gradient_accumulation_steps = 8,  # eff. batch 8
        num_train_epochs = 1,                       # ~325 steps (not max_steps)
        learning_rate = 2e-4, warmup_ratio = 0.03,  # scales warmup to run length
        logging_steps = 5, optim = "adamw_8bit", weight_decay = 0.01,
        save_steps = 100, save_total_limit = 2,     # checkpoint the multi-hour run
        output_dir = "outputs", report_to = "none",
        remove_unused_columns = False, dataset_kwargs = {"skip_prepare_dataset": True},
        max_seq_length = 2048,               # Qwen3-VL-8B's max in Unsloth (4096 auto-reduces)
    ),
)
trainer.train()

## Step 4 — sanity-check on a holdout page
The image must go through the PROCESSOR (which builds pixel_values) — NOT as an
`images=` kwarg to generate (that raises "model_kwargs not used by the model: ['images']").

In [ ]:
FastVisionModel.for_inference(model)
ho = load_dataset("MitchMooney/para-vlm-extraction", split="holdout")[0]
prompt = text_of(ho["messages"], "user")
msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]

input_text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
inputs = tokenizer(ho["image"], input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens=1024,
                   streamer=TextStreamer(tokenizer, skip_prompt=True))
print("\n--- GOLD ---\n", text_of(ho["messages"], "assistant"))   # compare

## Step 5 — save + bring it home

In [ ]:
model.save_pretrained("qwen3vl-para-lora"); tokenizer.save_pretrained("qwen3vl-para-lora")
# zip + download, or push the adapter to the Hub:
model.push_to_hub("MitchMooney/qwen3vl-8b-para-lora", token="hf_write_xxx", private=True)